# 02 — Exploratory Capacity Analysis

Explore total system load, CBP/HHS composition, operational flows, growth, and
seasonal patterns using the production metric and visualization layers.

In [ ]:
from pathlib import Path
import sys
import numpy as np  # noqa: F401 -- shared setup; used by modeling notebooks
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR = PROJECT_ROOT / "output"
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)

In [ ]:
from app_utils import (
    CBP_COLUMN, DISCHARGE_COLUMN, HHS_COLUMN, INTAKE_COLUMN,
    NET_INTAKE_COLUMN, TOTAL_LOAD_COLUMN, TRANSFER_COLUMN,
)
from src.visualisation import VisualizationConfig, create_dashboard_figures

metrics = pd.read_csv(
    PROCESSED_DIR / "uac_capacity_metrics_daily.csv",
    parse_dates=["Date"],
).set_index("Date")
metrics[[TOTAL_LOAD_COLUMN, CBP_COLUMN, HHS_COLUMN, NET_INTAKE_COLUMN]].describe().round(2)

In [ ]:
monthly = metrics.resample("ME").agg({
    TOTAL_LOAD_COLUMN: ["mean", "max", "last"],
    INTAKE_COLUMN: "sum",
    TRANSFER_COLUMN: "sum",
    DISCHARGE_COLUMN: "sum",
    NET_INTAKE_COLUMN: "sum",
})
monthly.tail(12).round(2)

In [ ]:
peak_date = metrics[TOTAL_LOAD_COLUMN].idxmax()
print({
    "peak_load": int(metrics[TOTAL_LOAD_COLUMN].max()),
    "peak_date": peak_date.date().isoformat(),
    "average_load": round(float(metrics[TOTAL_LOAD_COLUMN].mean()), 2),
    "positive_pressure_days": int(metrics[NET_INTAKE_COLUMN].gt(0).sum()),
})

In [ ]:
figures = create_dashboard_figures(
    metrics,
    VisualizationConfig(granularity="Monthly", show_anomalies=True),
    volatility_window_days=3,
)
figures.system_load

In [ ]:
figures.care_load_comparison

In [ ]:
figures.operational_flows

Monthly views sum operational flows and use period-end stock
values. Interpret apparent seasonal peaks alongside imputation and anomaly
flags, especially when comparing source reporting intervals.